# 033-class-loss-in-system: tuning

# Imports

In [ ]:
import itertools
from pathlib import Path

import pandas as pd

from uavloc.simulation.arms import FilterParams, HeadSpec, build_correlator, estimator_factory
from uavloc.simulation.sequence import flight_metrics, load_flight_set, per_flight_table, run_flights
from uavloc.track import REPO_ROOT, experiment_dir, set_seed

# Params

In [ ]:
EXP = "033-class-loss-in-system"
ARM = "control"                   # 033 arm: control | building | structure | noise
SEED = 42
DEVICE = "cpu"                    # CPU-only process: MPS corrupts CPU conv2d in the same process (bug-121)
ROUTE_SET = "sim-flights"
SPLIT = "tuning"
INITIAL_ERROR_M = 0.0             # normal start
SIGMA0_M = 1.0
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"   # relative to src/
SEG_ARCHITECTURE = "lraspp"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385
CLASS_WEIGHTS = [0.0, 1.0, 1.0, 0.5, 0.0, 0.5]
SEARCH_SCALE = 1.7
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0
FILTER = "particles"
N_PARTICLES = 2000
K_MAX_HYPOTHESES = 5
PEAKS_PER_HYPOTHESIS = 3
PEAK_NMS_M = 10.0
MERGE_DISTANCE_M = 5.0
CHILD_FLOOR_SIGMA_M = 1.0
WINDOW_SIGMAS = 3.0
WINDOW_CAP_HALF_M = 100.0
WINDOW_MARGIN_M = 2.0
HEAD_PATH = None                  # head .pt relative to src/; None: this experiment's artifacts/head-<ARM>.pt
HEAD_ENCODER_CHANNELS = [32, 32, 16]
HEAD_REFINE_CHANNELS = 16
TUNING_DIR = None                 # relative to the experiment folder; None: artifacts/tuning-<ARM>
TEMPERATURES = [0.25, 0.5, 1.0, 2.0]
Q_POS_M_PER_SQRT_S = 2.0          # fixed, as for 027.12
SHARD = 0
N_SHARDS = 2

In [ ]:
set_seed(SEED)

# Data

In [ ]:
set_config, specs = load_flight_set(ROUTE_SET, split=SPLIT)
len(specs)

# Model

In [ ]:
CORRELATOR = {
    "method": "segmentation", "seg_architecture": SEG_ARCHITECTURE,
    "seg_checkpoint_drone": SEG_CHECKPOINT, "seg_checkpoint_sat": SEG_CHECKPOINT,
    "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": MATCH_GSD_M, "match_gsd_m": MATCH_GSD_M, "matching": "weighted-ncc",
    "class_weights": CLASS_WEIGHTS, "window": "rect", "search_scale": SEARCH_SCALE, "fov_w_deg": FOV_W_DEG,
    "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
}
HEAD = HeadSpec(REPO_ROOT / (HEAD_PATH or f"experiments/{EXP}/artifacts/head-{ARM}.pt"),
                tuple(HEAD_ENCODER_CHANNELS), HEAD_REFINE_CHANNELS)
HEAD.checkpoint.exists()

In [ ]:
def filter_params(temperature: float, q_pos: float) -> FilterParams:
    return FilterParams(temperature, q_pos, N_PARTICLES, K_MAX_HYPOTHESES, PEAKS_PER_HYPOTHESIS, PEAK_NMS_M,
                        MERGE_DISTANCE_M, CHILD_FLOOR_SIGMA_M, WINDOW_SIGMAS, WINDOW_CAP_HALF_M, WINDOW_MARGIN_M)

In [ ]:
def correlator_for(map_raster):
    return build_correlator(map_raster, CORRELATOR, DEVICE, HEAD)

In [ ]:
combos = TEMPERATURES[SHARD::N_SHARDS]
factories = {f"{FILTER}|{t}|{Q_POS_M_PER_SQRT_S}": estimator_factory(FILTER, filter_params(t, Q_POS_M_PER_SQRT_S)) for t in combos}
list(factories)

# Run

In [ ]:
outcomes = run_flights(set_config, specs, INITIAL_ERROR_M, SIGMA0_M, correlator_for, factories)

In [ ]:
summary = pd.DataFrame([{"filter": name.split("|")[0], "temperature": float(name.split("|")[1]),
                         "q_pos_m_per_sqrt_s": float(name.split("|")[2]), **flight_metrics(found)}
                        for name, found in outcomes.items()])
summary.sort_values("mean_end_error_m")

# Save

In [ ]:
flights = pd.concat([per_flight_table(found).assign(combo=name) for name, found in outcomes.items()], ignore_index=True)
out = experiment_dir(EXP) / (TUNING_DIR or f"artifacts/tuning-{ARM}")
out.mkdir(parents=True, exist_ok=True)
summary.to_csv(out / f"shard-{SHARD}.csv", index=False)
flights.to_csv(out / f"shard-{SHARD}-flights.csv", index=False)